Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS: Literature-Driven Causal Normalization

Two new mechanisms,4fixed policies×2seeds×10fixed8/2folds. Source-only inner selection/refit. First-session equal-person scoring. Compare standalone and fixed10/20% additions to already trained8person full ensemble. Private T4;120min application cap. No competition submission or old baseline refit.

In [ ]:
import os,sys,pathlib,io,zipfile,base64,subprocess
os.environ.update(OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',MPLCONFIGDIR='/tmp/literature-mpl',XDG_CACHE_HOME='/tmp/literature-cache')
work=pathlib.Path('/tmp/proteus-literature-wave');work.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text()))) as z:z.extractall(work)
envdir=pathlib.Path('/tmp/literature-env')
subprocess.run([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(envdir)],check=True)
python=str(envdir/'bin/python')
subprocess.run([sys.executable,'-m','pip','--python',python,'install','--quiet','numpy==2.2.6','scipy==1.15.3','scikit-learn==1.7.2','pandas==2.3.3','pytest'],check=True)
out=pathlib.Path('/kaggle/working/literature-wave-results');out.mkdir(parents=True,exist_ok=True)
print((work/'PROTOCOL.md').read_text(),flush=True)
subprocess.run([python,'-m','pytest','-q','test_novel.py','--tb=short'],cwd=work,check=True)


In [ ]:
import signal,json,time
started=time.monotonic();expired=False
with (out/'execution.log').open('w') as log:
    process=subprocess.Popen([python,'-u','gpu.py','--minutes','110'],cwd=work,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    print('Training started; progress and curves saved in output directory.',flush=True)
    try:
        code=process.wait(timeout=115*60)
    except subprocess.TimeoutExpired:
        expired=True
        try:os.killpg(process.pid,signal.SIGTERM)
        except ProcessLookupError:pass
        try:process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try:os.killpg(process.pid,signal.SIGKILL)
            except ProcessLookupError:pass
            process.wait()
        code=process.returncode
(out/'watchdog.json').write_text(json.dumps(dict(expired=expired,returncode=code,seconds=time.monotonic()-started)))
if expired:
    # No refits: summarize only already completed checkpoints/predictions.
    subprocess.run([python,'-u','gpu.py','--combine-only'],cwd=work,timeout=290,check=True)
elif code:raise RuntimeError(f'Experiment exit code {code}; completed artifacts retained')


In [ ]:
print((out/'validation/REPORT.md').read_text() if (out/'validation/REPORT.md').exists() else 'See progress.json and failures')